In [ ]:
from langchain_ollama.chat_models import ChatOllama
from IPython.display import display, Markdown

llm = ChatOllama(model="huggingface.co/unsloth/Llama-3.2-3B-Instruct-GGUF:latest")

In [ ]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
    MessagesPlaceholder # NEW
)

from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
)

## Memory with LangChain

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content="You are a helpful assistant. Answer all questions to the best of your ability."),
        MessagesPlaceholder(variable_name="message"),
        HumanMessage(content="Can you repeat exactly your answer?")
    ]
)

variables = {
        "message": [
            HumanMessage(content="What does IU stand for as in university?"),
            AIMessage(content="Indiana University.")
        ],
    }

In [ ]:
prompt.invoke(variables)

In [ ]:
chain = prompt | llm
response = chain.invoke(variables)
Markdown(response.content)

---

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 1. Define your base prompt
prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content="You are a helpful assistant. Answer all questions clearly."),
        MessagesPlaceholder(variable_name="history"),   # stores prior messages
        MessagesPlaceholder(variable_name="input"),     # current turn
    ]
)

# 3. Create a simple chain
chain = prompt | llm

# 4. Define a memory store
# Here we use a simple Python dict to simulate per-session memory.
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 5. Wrap your chain with message history management
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history=get_session_history,
    history_messages_key="history",   # matches MessagesPlaceholder
    input_messages_key="input",       # matches MessagesPlaceholder
)

# 6. Simulate a conversation session
session_id = "user123"

In [ ]:
# First turn
response1 = chain_with_history.invoke(
    {"input": [HumanMessage(content="What does IU stand for as in university?")]},
    config={"configurable": {"session_id": session_id}},
)
print(response1.content)

In [ ]:
for msg in store['user123'].messages:
    print(msg)
    print()

In [ ]:
# Second turn — the model now sees previous context
response2 = chain_with_history.invoke(
    {"input": [HumanMessage(content="Can you repeat exactly your answer?")]},
    config={"configurable": {"session_id": session_id}},
)
print(response2.content)

In [ ]:
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

trimmer = trim_messages(
    strategy="last",
    max_tokens=2,
    token_counter=len,
    include_system=True
)


chain = (
    RunnablePassthrough.assign(
        history=lambda x: trimmer.invoke(x["history"])
    )
    | prompt
    | llm
)

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history=get_session_history,
    history_messages_key="history",   # matches MessagesPlaceholder
    input_messages_key="input",       # matches MessagesPlaceholder
)

In [ ]:
questions = [
    "Hey, my name is James",
    "What should I eat when the weather is hot?",
    "What does IU stand for as in university?"
]

In [ ]:
for question in questions:
    chain_with_history.invoke(
        {"input": [HumanMessage(content=question)]},
        config={"configurable": {"session_id": "user145"}},
    )
    res = chain_with_history.invoke(
        {"input": [HumanMessage(content="What is my name?")]},
        config={"configurable": {"session_id": "user145"}},
    )
    print(res)
    print()

#### Summary Memory

In [ ]:
chat_history = [
    HumanMessage(content="Hey there! I'm a student at IU."),
    AIMessage(content="Hello!"),
    HumanMessage(content="How are you today?"),
    AIMessage(content="Fine thanks!"),
    HumanMessage(content="Do you know anything about IU?"),
    AIMessage(content="Indiana University is in Bloomingon, Indiana")
]

In [ ]:
summary_prompt = (
    "Distill the above chat messages into a single summary message. "
    "Include as many specific details as you can."
)
summary_message = llm.invoke(
    chat_history + [HumanMessage(content=summary_prompt)]
)

In [ ]:
Markdown(summary_message.content)

In [ ]:
new_turns = [
    HumanMessage(content="What programs is IU known for?"),
    AIMessage(content="It’s well known for business, music, and computer science programs."),
]

In [ ]:
# more concrete implementation
def summarize_history_if_long(history, llm, summary_message=None, max_len=3):
    """If the history is long, summarize it into one message."""
    print(len(history))
    if len(history) > max_len:
        summary_prompt = (
            "Summarize the following conversation so far into one concise message.\n"
            "Focus on key facts, roles, and context.\n\n"
            f"{[m.content for m in history]}"
        )
        summary_message = llm.invoke([HumanMessage(content=summary_prompt)])
        # return a new short history that keeps only summary
        print("HISTORY:", history)
        return [AIMessage(content=summary_message.content)]

    return history

In [ ]:
chain = (
    RunnablePassthrough.assign(
        history=lambda x: summarize_history_if_long(x["history"], llm)
    )
    | prompt
    | llm
)

In [ ]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history=get_session_history,
    history_messages_key="history",
    input_messages_key="input",
)

In [ ]:
for question in questions:
    chain_with_history.invoke(
        {"input": [HumanMessage(content=question)]},
        config={"configurable": {"session_id": "user170"}},
    )
    res = chain_with_history.invoke(
        {"input": [HumanMessage(content="What is my name?")]},
        config={"configurable": {"session_id": "user170"}},
    )
    print(res)
    print()

In [ ]:
for i in chain_with_history.get_session_history('user170').messages:
    print(i)

In [ ]:
# TODO # change the code such that we combine summary + last few messages